# **Hybrid Search**

### hybrid search is the combinasion between the vector search that we already saw with embeddings, and keyword search that is traditionally used to match exactly the text, with both having some weaknesses, the mix of them creates a very powerfully search

In [46]:
from langchain_community.retrievers import  BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_chroma import Chroma
from langchain_core.documents import Document
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_cohere import CohereRerank
from groq import Groq
load_dotenv()

True

In [47]:
chunks = [
    # Tesla - Financial & Production
    "Tesla reported record quarterly revenue of $25.2 billion in Q3 2024.",
    "Tesla's automotive gross margin improved to 19.3% this quarter.",
    "Tesla Cybertruck production ramp begins in 2024 with initial deliveries.",
    "Tesla announced plans to expand Gigafactory production capacity.",
    "Tesla stock price reached new highs following earnings announcement.",
    "Tesla's energy storage business grew 40% year-over-year.",
    "Tesla continues to lead in electric vehicle market share globally.",
    "Tesla Model Y became the best-selling vehicle worldwide.",
    "Tesla reported strong free cash flow generation of $7.5 billion.",
    "Tesla's Full Self-Driving revenue increased significantly.",
    
    # Microsoft - Development & Acquisitions
    "Microsoft acquired GitHub for $7.5 billion in 2018.",
    "Microsoft's cloud revenue Azure grew 29% year-over-year.",
    "Microsoft announced new AI features for Visual Studio Code.",
    "Microsoft Teams integration with GitHub enhances developer workflow.",
    "Microsoft's developer tools division sees strong adoption.",
    "Microsoft acquired Activision Blizzard for $68.7 billion.",
    "Microsoft's productivity suite gained 50 million new users.",
    "Microsoft announced new Surface devices for developers.",
    "Microsoft's AI Copilot features expand to more development tools.",
    "Microsoft's enterprise solutions drive revenue growth.",
    
    # NVIDIA - AI & Hardware
    "NVIDIA's data center revenue reached $47.5 billion annually.",
    "NVIDIA's H100 GPUs see unprecedented demand for AI training.",
    "NVIDIA announced next-generation Blackwell architecture.",
    "NVIDIA's gaming revenue declined due to crypto market changes.",
    "NVIDIA's automotive AI platform partnerships expanded.",
    "NVIDIA's AI chip shortage affects cloud providers.",
    "NVIDIA stock valuation exceeds $2 trillion market cap.",
    "NVIDIA's CUDA platform dominates AI development.",
    "NVIDIA announced new AI inference chips for edge computing.",
    "NVIDIA's partnership with major cloud providers strengthens.",
    
    # Google/Alphabet - AI & Cloud
    "Google's AI investments total over $100 billion in recent years.",
    "Google Cloud revenue grew 35% reaching $8.4 billion quarterly.",
    "Google announced Gemini AI model competing with GPT-4.",
    "Google's search advertising revenue remains strong at $59 billion.",
    "Google's Workspace products integrate advanced AI features.",
    "Google announced quantum computing breakthroughs.",
    "Google's autonomous vehicle division Waymo expands operations.",
    "Google's AI research published breakthrough papers.",
    "Google's cloud AI services see enterprise adoption.",
    "Google faces regulatory scrutiny over AI dominance.",
    
    # Noisy/Less Relevant Chunks
    "The Tesla coil was invented by Nikola Tesla in 1891.",
    "Microsoft Excel spreadsheet formulas can be complex for beginners.",
    "NVIDIA Shield TV streaming device gets software update.",
    "Google Maps navigation improved with real-time traffic data.",
    "Production delays affected multiple manufacturing sectors.",
    "Financial markets showed volatility during earnings season.",
    "Revenue recognition standards changed for software companies.",
    "Hardware components face supply chain constraints globally.",
    "Development tools market grows with remote work trends.",
    "AI research requires significant computational resources.",
    "Quarterly reports show mixed results across tech sector.",
    "Stock market analysts upgrade technology sector ratings.",
    "Cloud computing adoption accelerates in enterprise market.",
    "Data center construction increases globally.",
    "Semiconductor shortage impacts various industries.",
    "Electric vehicle charging infrastructure expands rapidly.",
    "Software development productivity tools gain popularity.",
    "Machine learning frameworks become more accessible.",
    "Enterprise software licensing models evolve.",
    "Technology conferences showcase latest innovations."
]

In [48]:
documents = [Document(page_content=chunk,metadata={"source":f"chunk {i+1}"}) for i,chunk in enumerate(chunks)]
documents[0]

Document(metadata={'source': 'chunk 1'}, page_content='Tesla reported record quarterly revenue of $25.2 billion in Q3 2024.')

## **Retrivers**

### **1-vector retriever**

In [49]:
embedding_model = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
dir = "../db/hybrid_db"
vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_metadata={"hsnw:space":"cosine"}
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7997.11it/s]


In [50]:
vector_retriever = vector_store.as_retriever(search_kwargs={"k":15})
test_vector_query = "space exploration company"
relevent_docs = vector_retriever.invoke(test_vector_query)
for doc in relevent_docs:
    print(doc.page_content)

SpaceX develops Starship rockets for Mars missions.
Google is a large technology company with global operations.
NVIDIA designs Starship architecture for their new GPUs.
Microsoft is a large technology company with global operations.
Microsoft acquired GitHub for 7.5 billion dollars in 2018.
Microsoft acquired GitHub for $7.5 billion in 2018.
Microsoft acquired GitHub for $7.5 billion in 2018.
Microsoft acquired GitHub for $7.5 billion in 2018.
Data center construction increases globally.
Data center construction increases globally.
Data center construction increases globally.
Semiconductor shortage impacts various industries.
Semiconductor shortage impacts various industries.
Semiconductor shortage impacts various industries.
Tesla announced plans to expand Gigafactory production capacity.


### **2-BM25 retriever**

In [51]:
bm25_retriver = BM25Retriever.from_documents(documents)
bm25_retriver.k = 15

In [52]:
test_keyword_query = "Tesla"
relevent_docs_keyword = bm25_retriver.invoke(test_keyword_query)
for rel in relevent_docs_keyword:
    print(rel.page_content)

The Tesla coil was invented by Nikola Tesla in 1891.
Tesla Model Y became the best-selling vehicle worldwide.
Tesla announced plans to expand Gigafactory production capacity.
Tesla stock price reached new highs following earnings announcement.
Tesla reported strong free cash flow generation of $7.5 billion.
Tesla Cybertruck production ramp begins in 2024 with initial deliveries.
Tesla continues to lead in electric vehicle market share globally.
Tesla reported record quarterly revenue of $25.2 billion in Q3 2024.
Electric vehicle charging infrastructure expands rapidly.
Technology conferences showcase latest innovations.
Machine learning frameworks become more accessible.
Enterprise software licensing models evolve.
Development tools market grows with remote work trends.
Hardware components face supply chain constraints globally.
Financial markets showed volatility during earnings season.


## **3-hybrid search**

In [53]:
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriver,vector_retriever],
    weights=[0.5,0.5]
)

In [54]:
hybrid_test_query = "purchase cost 7.5 billion"

relevent_docs_hybrid = hybrid_retriever.invoke(hybrid_test_query)

for rel in relevent_docs_hybrid:
    print(rel.page_content)

Microsoft acquired GitHub for $7.5 billion in 2018.
Tesla reported strong free cash flow generation of $7.5 billion.
NVIDIA stock valuation exceeds $2 trillion market cap.
NVIDIA's data center revenue reached $47.5 billion annually.
Microsoft acquired Activision Blizzard for $68.7 billion.
Google Cloud revenue grew 35% reaching $8.4 billion quarterly.
Google's AI investments total over $100 billion in recent years.
Microsoft acquired GitHub for 7.5 billion dollars in 2018.
Tesla reported record quarterly revenue of $25.2 billion in Q3 2024.
Electric vehicle charging infrastructure expands rapidly.
Data center construction increases globally.
Semiconductor shortage impacts various industries.
Technology conferences showcase latest innovations.
Enterprise software licensing models evolve.
Machine learning frameworks become more accessible.
Software development productivity tools gain popularity.
Hardware components face supply chain constraints globally.
Revenue recognition standards cha

# **Reranker**

What is it?

A reranker is a specialized AI model that acts as a "quality inspector" that improves the initial search results by
reordering them based on semantic relevance to your query.

Think of it as a "second opinion" system. Your initial retrieval (result of hybrid + RRF) might return 100 chunks.

The reranker takes these chunks and applies more sophisticated analysis to reorder them by putting the most relevant
chunks at the top.

How does reranker (cross-encoder) achieve this ?:

Embeddings (Bi-encoder approach):

Query: "apple stock price" - [0.1, 0.8, 0.2, ... ] (vector)
Chunk: "Apple trees grow in orchards" - [0.3, 0.7, 0.1, ... ] (vector)
Similarity = cosine(query_vector, chunk_vector) = 0.78

In [55]:
import os
COHERE_API_KEY = os.getenv("COHERE_API_KEY")
reranker = CohereRerank(model="rerank-english-v3.0",top_n=10,cohere_api_key=COHERE_API_KEY)
reranked_docs = reranker.compress_documents(relevent_docs_hybrid,hybrid_test_query)


In [56]:
for i,doc in enumerate(reranked_docs):
    print("document")
    print(doc.page_content)

document
Microsoft acquired GitHub for $7.5 billion in 2018.
document
Microsoft acquired GitHub for 7.5 billion dollars in 2018.
document
Tesla reported strong free cash flow generation of $7.5 billion.
document
Microsoft acquired Activision Blizzard for $68.7 billion.
document
Google's AI investments total over $100 billion in recent years.
document
NVIDIA's data center revenue reached $47.5 billion annually.
document
Financial markets showed volatility during earnings season.
document
Google Cloud revenue grew 35% reaching $8.4 billion quarterly.
document
Semiconductor shortage impacts various industries.
document
Hardware components face supply chain constraints globally.


## **LLM**

In [ ]:

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
prompt = f"""answer the following question based on the documents provided, query : {hybrid_test_query}
    Documents : 
    {chr(10).join([doc.page_content for doc in relevent_docs_hybrid])}
    provide an answer only based on the documents, if you dont find the  answer in them, just so no enough informations for the question
"""
final_prompt = [{
    "role":"user",
    "content":prompt
}]
client = Groq(api_key=GROQ_API_KEY)
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    temperature=0,
    messages=final_prompt
)
print(response.choices[0].message.content)

Microsoft acquired GitHub for a purchase cost of 7.5 billion dollars in 2018.
